In [1]:

from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))


c:\Users\cayog\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [3]:
from src.tools import read_calendar, get_customer_profile

tools = {
    "read_calendar": read_calendar,
    "get_customer_profile": get_customer_profile
}


In [4]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
Classify this email into one of:
ignore
notify_human
respond
Email:
{email}
Return only the label.
"""
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}


In [5]:
def react_agent(state):
    email = state["email"]
    prompt = f"""
You are an email assistant.
You can use tools if needed.
Tools:
read_calendar
get_customer_profile
Email:
{email}
If you need a tool, write TOOL:<toolname>
Otherwise give reply.
"""
    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}

    return {**state, "response": response}


In [6]:
graph = StateGraph(dict)

graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)

def route(state):
    if state["triage"] == "respond":
        return "react"
    else:
        return "end"

graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")

app = graph.compile()


In [10]:
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

results = []
for _, row in emails.head(15).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})
    print(output["triage"])
    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })

pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)


Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


notify_human


Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


notify_human


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 22.199956903s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
import pandas as pd

emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

gold = pd.DataFrame({
    "email": emails.head(25)["body"],
    "expected": [""] * 25
})

gold.to_csv("../data/golden_labels.csv", index=False)


In [ ]:
gold = pd.read_csv("../data/golden_labels.csv")
pred = pd.read_csv("../data/milestone1_output.csv")

gold = gold.reset_index(drop=True)
pred = pred.reset_index(drop=True)

accuracy = (gold["expected"] == pred["triage"]).mean()
accuracy


0.9